# Transformar Datos Job Position
- Preprocesar la cadena Json para poder corregir los problemas de calidad de datos
- Transfomar la cadena Json en un Objeto Json
- Escribir los datos trasnsformados en el schema silver job_positions_json
- Acceder a los elementos del objeto JSON
- Eliminar los elementos duplicados 
- Anular el anidamiento de la matriz
- Escribir los datos en el schema silver 

In [0]:
SELECT *FROM zenviro.bronze.v_job_positions limit 10

In [0]:
SELECT 
regexp_replace(value,'"title": ([^",}]+)','"title": "\$1"') as fixed_value
FROM zenviro.bronze.v_job_positions 

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_job_positions AS
SELECT 
regexp_replace(value,'"title": ([^",}]+)','"title": "\$1"') as fixed_value
FROM zenviro.bronze.v_job_positions 

In [0]:
SELECT 
schema_of_json(fixed_value)
FROM tv_job_positions limit 1

In [0]:
SELECT 
    from_json(fixed_value,
    'STRUCT<description: STRING, job_position_id: BIGINT, salary: ARRAY<STRUCT<salary_max: BIGINT, salary_min: BIGINT>>, title: STRING>'
    ) AS json_value
FROM tv_job_positions 

In [0]:
CREATE OR REPLACE TABLE zenviro.silver.job_positions_json
AS
SELECT 
    from_json(fixed_value,
    'STRUCT<description: STRING, job_position_id: BIGINT, salary: ARRAY<STRUCT<salary_max: BIGINT, salary_min: BIGINT>>, title: STRING>'
    ) AS json_value
FROM tv_job_positions 

In [0]:
SELECT *FROM zenviro.silver.job_positions_json limit 10

In [0]:
SELECT 
    json_value.job_position_id,
    json_value.title,
    json_value.description,
    array_distinct(json_value.salary)
FROM zenviro.silver.job_positions_json 

In [0]:
SELECT 
    json_value.job_position_id,
    json_value.title,
    json_value.description,
    explode(array_distinct(json_value.salary)) AS salary
FROM zenviro.silver.job_positions_json 

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_job_positions_01 AS
SELECT 
    json_value.job_position_id,
    json_value.title,
    json_value.description,
    explode(array_distinct(json_value.salary)) AS salary
FROM zenviro.silver.job_positions_json 

In [0]:
SELECT 
job_position_id,
title,
description,
salary.salary_max,
salary.salary_min
FROM tv_job_positions_01

In [0]:
CREATE TABLE IF NOT EXISTS zenviro.silver.job_positions 
AS
SELECT 
job_position_id,
title,
description,
salary.salary_max,
salary.salary_min
FROM tv_job_positions_01
 